# 🏆 WAXAL ASR Challenge — Final Submission
**Author:** Bright Sikazwe
**Target Languages:** Lingala (lin), Shona (sna), Luganda (lug)
**Evaluation Metric:** 0.5 * WER + 0.5 * CER

## 📑 Zindi Code Review Documentation

### 1. Overview and Objectives
This notebook implements an automated, end-to-end Automatic Speech Recognition (ASR) fine-tuning pipeline. The objective is to maximize transcription accuracy for three low-resource African languages using OpenAI's `whisper-large-v3-turbo` architecture. To accommodate compute constraints while maximizing performance, the model is fine-tuned using **Low-Rank Adaptation (LoRA)**.

### 2. Environment & Reproducibility
* **Hardware:** Google Colab Pro (NVIDIA A100 / L4 GPU Premium class).
* **Randomness:** Global random seeds are pinned to `42` across Python, NumPy, and PyTorch to guarantee reproducible weight initialization and data shuffling.
* **Libraries:** Specific dependencies include `transformers`, `peft`, `datasets`, `librosa`, and `jiwer`. A targeted `torchao>=0.16.0` install is used to ensure compatibility with modern PEFT adapters.

### 3. Architecture & ETL Pipeline
**Extract:**
* Metadata (`Train.csv`, `Test.csv`, `SampleSubmission.csv`) is loaded from Google Drive.
* Audio arrays are extracted from the Hugging Face hub (`google/WaxalNLP`). **Design Choice:** To prevent disk overflow (OOM) from the 100GB+ unlabeled split, the pipeline uses a targeted `data_files` fetch to *only* pull the strictly necessary `train`, `validation`, and `test` parquet files.

**Transform:**
* **Data Cleansing:** Malformed rows in the raw CSVs (e.g., unescaped quotes causing column shifts) are dynamically dropped using `on_bad_lines='skip'`.
* **Zero Leakage Rule:** Text transcriptions are programmatically stripped from the `test` dataset object immediately upon extraction to guarantee zero label leakage.
* **Audio Resampling:** The PyTorch `WaxalDataset` class processes audio on-the-fly, using `librosa` to resample varying sample rates strictly to Whisper's required 16,000 Hz. Audio exceeding 30 seconds is truncated.
* **Language Mapping:** Luganda (`lug`) is not natively supported by Whisper's tokenizer. Following cross-lingual transfer best practices, it is mapped to Swahili (`swa`).

**Load & Model:**
* Data is passed via a custom PyTorch collator that dynamically tokenizes text and extracts log-mel spectrograms.
* The `whisper-large-v3-turbo` model is loaded in `FP16`.
* **PEFT/LoRA:** We inject adapter weights (Rank=32, Alpha=64) into the `q_proj`, `v_proj`, `k_proj`, and `out_proj` attention modules.

### 4. Inference & Performance Metrics
* **Inference:** The model utilizes batch generation (`max_new_tokens=200`).
* **Metrics:** Out-of-band evaluation calculates both raw and normalized Word Error Rate (WER) and Character Error Rate (CER) using the `jiwer` library. Text normalization removes punctuation and standardizes casing before scoring.

### 5. Error Handling & Maintenance
* **Fault Tolerance:** The `Seq2SeqTrainer` is configured to search the designated `work/` directory for recent checkpoints. If the compute instance disconnects, re-running the cell will automatically resume training from the last saved state.

**INSTALL LIBS**

In [1]:
%%capture
!apt-get update -y -q
!apt-get install -y -q ffmpeg
!pip install -q -U transformers datasets peft accelerate jiwer soundfile librosa evaluate bitsandbytes "torchao>=0.16.0"

**MAIN CODE**

In [2]:
ZINDI_DIR = "/content/drive/MyDrive/waxal"

In [3]:
# ==============================================================================
# ZINDI COMPETITION CODE SUBMISSION DOCUMENTATION
# Challenge: WAXAL ASR Challenge
# Architecture: V2 Production Pipeline (Teacher-Student Distillation)
# Upgrades: Rank 64 LoRA, Cosine Decay, Beam Search, FLEURS Integration, Auto-Resume
# ==============================================================================

import os, random, numpy as np, torch
import pandas as pd
import datasets
from datasets import load_dataset, concatenate_datasets, Audio
import librosa
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
import jiwer
import re
import gc
import shutil
import glob

# --- 1. Reproducibility Anchor ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# --- 2. V2 Hyperparameters & Configuration ---
MODEL_ID   = "openai/whisper-large-v3-turbo"
LANGS = ["lin", "sna", "lug"]
WHISPER_LANG = {"lin": "lingala", "sna": "shona", "lug": "swahili"}

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/waxal"
os.makedirs(DRIVE_DIR, exist_ok=True)
WORK_DIR  = f"{DRIVE_DIR}/work"
ZINDI_DIR = DRIVE_DIR

# 🚀 V2 UPGRADES
MAX_AUDIO_SECONDS = 30.0
NUM_EPOCHS        = 5
BATCH_SIZE        = 16
GRAD_ACCUM        = 4
LR                = 1e-4

from google.colab import userdata
from huggingface_hub import login
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

ACTIVE_GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Hardware Runtime: {ACTIVE_GPU} | V2 Pipeline Initiated")

def text_normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    return re.sub(r"\s+", " ", text)

# ==============================================================================
# ## 3. DATA ENGINEERING: EXTERNAL MOATS & PSEUDO-LABELING
# ==============================================================================
print("\n[Data Engineering] Constructing Synthetic and External Data Moats...")
processor = WhisperProcessor.from_pretrained(MODEL_ID, task="transcribe")

def fetch_fleurs_data(samples_per_lang=400):
    fleurs_recs = []
    fleurs_map = {"lin": "ln_cd", "sna": "sn_zw", "lug": "lg_ug"}

    for zindi_lang, fleurs_lang in fleurs_map.items():
        print(f"  -> Pulling External FLEURS Data for {zindi_lang.upper()}...")
        try:
            ds = load_dataset("google/fleurs", fleurs_lang, split=f"train[:{samples_per_lang}]")
            ds = ds.cast_column("audio", Audio(sampling_rate=16000))

            text_col = [c for c in ds.column_names if "transcript" in c.lower() or "text" in c.lower() or "sentence" in c.lower()][0]

            for i, row in enumerate(ds):
                arr = np.asarray(row["audio"]["array"], dtype=np.float32)
                fleurs_recs.append({
                    "id": f"fleurs_{zindi_lang}_{i}",
                    "lang": zindi_lang,
                    "text": text_normalize(row[text_col]),
                    "array": arr[: int(MAX_AUDIO_SECONDS * 16000)],
                    "src": "memory"
                })
        except Exception as e:
            print(f"     [Warning] Skipped FLEURS {zindi_lang}: {e}")
    return fleurs_recs

def generate_pseudo_labels(v1_model_path, samples_per_lang=300):
    if not os.path.exists(v1_model_path):
        print("  -> V1 Model not found. Skipping pseudo-labeling.")
        return []

    print(f"  -> Loading V1 Teacher Model from {v1_model_path}...")
    from peft import PeftModel
    base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True)
    v1_teacher = PeftModel.from_pretrained(base, v1_model_path).eval().cuda()

    pseudo_recs = []
    for lang in LANGS:
        print(f"  -> Pseudo-labeling {samples_per_lang} raw files for {lang.upper()}...")
        try:
            file_pattern = f"data/ASR/{lang}/{lang}-unlabeled-00000.parquet"
            ds = load_dataset("google/WaxalNLP", data_files={"unlabeled": file_pattern}, split=f"unlabeled[:{samples_per_lang}]")
            ds = ds.cast_column("audio", Audio(sampling_rate=16000))

            for i, row in enumerate(ds):
                arr = np.asarray(row["audio"]["array"], dtype=np.float32)
                feats = processor.feature_extractor(arr, sampling_rate=16000, return_tensors="pt").input_features.half().cuda()
                with torch.no_grad():
                    gen_ids = v1_teacher.generate(feats, language=WHISPER_LANG[lang], task="transcribe", max_new_tokens=100)
                text = processor.decode(gen_ids[0], skip_special_tokens=True).strip()

                pseudo_recs.append({
                    "id": f"pseudo_{lang}_{i}", "lang": lang, "text": text,
                    "array": arr[: int(MAX_AUDIO_SECONDS * 16000)], "src": "memory"
                })
        except Exception as e:
            print(f"     [Warning] Skipped Pseudo-labeling {lang}: {e}")

    del v1_teacher, base
    torch.cuda.empty_cache()
    gc.collect()
    return pseudo_recs

external_recs = fetch_fleurs_data()
pseudo_recs = generate_pseudo_labels(f"{WORK_DIR}/final_lora")

# ==============================================================================
# ## 4. WAXAL BASE ETL & DATASET MERGING
# ==============================================================================
print("\n[ETL Stage] Loading Official Zindi Datasets...")

train_df = pd.read_csv(f"{ZINDI_DIR}/Train.csv", dtype=str, keep_default_na=False, on_bad_lines='skip')
test_df  = pd.read_csv(f"{ZINDI_DIR}/Test.csv",  dtype=str)
sub_df   = pd.read_csv(f"{ZINDI_DIR}/SampleSubmission.csv", dtype=str)

train_df = train_df[train_df["language"].isin(LANGS)].copy()
train_df = train_df[train_df["transcription"].str.strip().ne("")]
test_df["language"] = test_df["ID"].str.split("_").str[0]

def load_base_waxal(lang: str):
    parts = []
    for split in ["train", "validation", "test"]:
        try:
            ds = load_dataset("google/WaxalNLP", data_files={split: f"data/ASR/{lang}/{lang}-{split}-*.parquet"}, split=split)
            ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        except Exception: continue
        keep_columns = ["id", "audio"] + ([] if split == "test" else [c for c in ds.column_names if "transcript" in c.lower()])
        ds = ds.remove_columns([c for c in ds.column_names if c not in keep_columns])
        parts.append(ds)
    return concatenate_datasets(parts)

!rm -rf ~/.cache/huggingface/datasets/downloads
audio_ds = {lang: load_base_waxal(lang) for lang in LANGS}

id_to_idx = {}
for lang in LANGS:
    for i, _id in enumerate(audio_ds[lang]["id"]):
        id_to_idx[_id] = (lang, i)

def build_manifest(df: pd.DataFrame, with_text: bool = True) -> list:
    recs = []
    trans_map = dict(zip(train_df["id"], train_df["transcription"])) if with_text else {}
    for _, r in df.iterrows():
        _id = r.get("id", r.get("ID"))
        if _id not in id_to_idx: continue
        recs.append({
            "id": _id, "lang": id_to_idx[_id][0], "src_idx": id_to_idx[_id][1],
            "text": trans_map[_id] if with_text else "", "src": "waxal"
        })
    return recs

train_recs = build_manifest(train_df[train_df.original_split == "train"]) + external_recs + pseudo_recs
val_recs   = build_manifest(train_df[train_df.original_split == "validation"])
test_recs  = build_manifest(test_df, with_text=False)

print(f"\n[Manifest] Total V2 Training Samples: {len(train_recs)} (Includes {len(external_recs)} External, {len(pseudo_recs)} Synthetic)")

class V2WaxalDataset(torch.utils.data.Dataset):
    def __init__(self, recs, training=True):
        self.recs, self.training = recs, training
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        if r["src"] == "waxal":
            arr = np.asarray(audio_ds[r["lang"]][r["src_idx"]]["audio"]["array"], dtype=np.float32)
        else:
            arr = r["array"]

        output = {"array": arr[: int(MAX_AUDIO_SECONDS * 16000)], "lang": r["lang"], "id": r["id"]}
        if self.training: output["text"] = r["text"]
        return output

def make_collator(training=True):
    def collate(batch):
        feats = processor.feature_extractor([b["array"] for b in batch], sampling_rate=16000, return_tensors="pt")
        out = {"input_features": feats.input_features}
        if training:
            labels = []
            for b in batch:
                processor.tokenizer.set_prefix_tokens(language=WHISPER_LANG[b["lang"]], task="transcribe")
                labels.append(processor.tokenizer(b["text"]).input_ids)
            maxlen = min(max(len(l) for l in labels), 448)
            lab = torch.full((len(labels), maxlen), -100, dtype=torch.long)
            for i, l in enumerate(labels):
                l = l[:maxlen]
                lab[i, :len(l)] = torch.tensor(l)
            out["labels"] = lab
        return out
    return collate

# ==============================================================================
# ## 5. V2 HIGH-CAPACITY MODEL INITIALIZATION
# ==============================================================================
print("\n[V2 Architecture] Initializing High-Capacity LoRA Model...")
base_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True)
base_model.config.forced_decoder_ids = None
base_model.config.suppress_tokens = []

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
base_model = prepare_model_for_kbit_training(base_model)

lora = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"]
)
v2_model = get_peft_model(base_model, lora)
print("✅ V2 High-Capacity LoRA Adapters active.")

# ==============================================================================
# ## 6. V2 TRAINING CONFIGURATION
# ==============================================================================
args = Seq2SeqTrainingArguments(
    output_dir=f"{WORK_DIR}/whisper-waxal-lora-v2",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=50,
    save_steps=1000,
    save_total_limit=2,
    eval_strategy="no",
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=v2_model, args=args,
    train_dataset=V2WaxalDataset(train_recs), data_collator=make_collator(True)
)

print("\n[Training Phase] Commencing V2 Training Loop...")

# 🚀 FAULT TOLERANCE FIX: Search for checkpoints to automatically resume
ckpts = glob.glob(f"{WORK_DIR}/whisper-waxal-lora-v2/checkpoint-*")
resume_path = max(ckpts, key=os.path.getmtime) if ckpts else None

if resume_path:
    print(f"Resuming training from previous LoRA checkpoint: {resume_path}")
else:
    print("No checkpoints found. Starting fresh V2 LoRA training run.")

trainer.train(resume_from_checkpoint=resume_path)
trainer.save_model(f"{WORK_DIR}/final_lora_v2")

# ==============================================================================
# ## 7. V2 BEAM SEARCH INFERENCE & EVALUATION
# ==============================================================================
print("\n[Evaluation Phase] Running Inference with Beam Search Decoding...")
v2_model.eval().half().cuda()

@torch.no_grad()
def transcribe_inference_loop(manifest_recs, batch_size=16):
    dataset_eval = V2WaxalDataset(manifest_recs, training=False)
    predictions = {}
    for start in range(0, len(dataset_eval), batch_size):
        batch = [dataset_eval[i] for i in range(start, min(start + batch_size, len(dataset_eval)))]
        feats = processor.feature_extractor([b["array"] for b in batch], sampling_rate=16000, return_tensors="pt")

        gen_ids = v2_model.generate(
            feats.input_features.half().cuda(),
            language=WHISPER_LANG[batch[0]["lang"]], task="transcribe",
            max_new_tokens=200,
            num_beams=5,
            repetition_penalty=1.1,
            early_stopping=True
        )
        for b, text in zip(batch, processor.batch_decode(gen_ids, skip_special_tokens=True)):
            predictions[b["id"]] = text.strip()
        gc.collect()
    return predictions

for lang in LANGS:
    recs = [r for r in val_recs if r["lang"] == lang]
    preds = transcribe_inference_loop(recs)
    refs, hyps = [r["text"] for r in recs], [preds[r["id"]] for r in recs]
    norm_wer = jiwer.wer([text_normalize(x) for x in refs], [text_normalize(x) for x in hyps])
    print(f"[{lang.upper()}] V2 Normalized WER: {norm_wer:.4f}")

all_test_predictions = {}
for lang in LANGS:
    all_test_predictions.update(transcribe_inference_loop([r for r in test_recs if r["lang"] == lang]))

final_submission = sub_df.copy()
final_submission["Target"] = final_submission["ID"].map(all_test_predictions).str.strip().replace("", "a")
final_submission.to_csv(f"{WORK_DIR}/submission_v2.csv", index=False)
print(f"✅ Master Pipeline Complete. Zindi file ready: {WORK_DIR}/submission_v2.csv")

Mounted at /content/drive


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hardware Runtime: NVIDIA A100-SXM4-80GB | V2 Pipeline Initiated

[Data Engineering] Constructing Synthetic and External Data Moats...


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

...

[SNA] V2 Normalized WER: 0.2475


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

[LUG] V2 Normalized WER: 0.2018


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

✅ Master Pipeline Complete. Zindi file ready: /content/drive/MyDrive/waxal/work/submission_v2.csv
